# Variant 15. Fine tune with answer only. Drop MATH_ANSAUG

## Run

Kaggle: GPU ON, Internet OFF. Tổng thời gian dự kiến **<= 3 giờ**.

Output:
- `data/train_preprocessed.json`: train set sau preprocessing và smart truncation.
- `data/train_preprocessing_report.json`: thống kê preprocessing.
- `gpt2_math_baseline_ckpt/`: checkpoint sau fine-tune.
- `valid_output.json` + `valid_report.json`: output và đánh giá chi tiết validation.
- `test_predictions.json`: file nộp cho Phase 2.


In [ ]:
import os
import re
import gc
import json
import math
import random
import inspect
import zipfile
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

VARIANT_NAME = "v15_answer_only_drop_math_ansaug_clean40k"
SAFE_EOS_ID = 50256

# Kaggle official paths. Local fallback paths are included only for debugging outside Kaggle.
TRAIN_PATH_CANDIDATES = [
    Path("/kaggle/input/datasets/kimanh2002/dataset-math/train.json"),
    Path("/kaggle/input/dataset-math/train.json"),
    Path("/mnt/data/extracted/train.json"),
    Path("/mnt/data/train.json"),
]
VALID_PATH_CANDIDATES = [
    Path("/kaggle/input/datasets/kimanh2002/dataset-math/valid.json"),
    Path("/kaggle/input/dataset-math/valid.json"),
    Path("/mnt/data/extracted/valid.json"),
    Path("/mnt/data/valid.json"),
]
TEST_PATH_CANDIDATES = [
    Path("/kaggle/input/datasets/kimanh2002/dataset-math/test.json"),
    Path("/kaggle/input/dataset-math/test.json"),
    Path("/mnt/data/extracted/test.json"),
    Path("/mnt/data/test.json"),
]
MODEL_PATH_CANDIDATES = [
    Path("/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese"),
    Path("/kaggle/input/nlphustgpt2-vietnamese"),
    Path("/kaggle/input/nlp-hust-gpt2-vietnamese"),
    Path("/kaggle/input/gpt2-vietnamese"),
]

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/mnt/data")
PROC_DIR = WORK_DIR / f"processed_{VARIANT_NAME}"
MODEL_OUT_DIR = WORK_DIR / f"lora_{VARIANT_NAME}"
PRED_DIR = WORK_DIR / f"predictions_{VARIANT_NAME}"
for p in [PROC_DIR, MODEL_OUT_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# =========================
# Variant 15 data policy
# =========================
TRAIN_MAX_SAMPLES = 40000

# Hard exclude for training only. valid/test are NOT removed by these rules.
# Rationale:
# - MATH_AnsAug: observed weak type for this setup.
# - MATH_FOBAR/MATH_SV: symbolic synthetic inverse problems, often long/noisy for small GPT-2 answer-only.
# - GSM_FOBAR: inverse-variable generated questions, long and easier to pollute direct answer learning.
EXCLUDE_TRAIN_TYPES = {"MATH_AnsAug", "MATH_FOBAR", "MATH_SV", "GSM_FOBAR"}

# Balanced clean subset. If any pool is smaller after filtering, the notebook backfills from remaining clean records.
PREFERRED_TYPE_QUOTAS = {
    "GSM_Rephrased": 16500,
    "GSM_AnsAug": 6300,
    "MATH_Rephrased": 8000,
    "GSM_SV": 9200,
}

MAX_QUERY_CHARS = 900
MAX_ANSWER_CHARS = 48
DROP_QUERY_REGEX = [
    r"\\begin\s*\{",       # heavy latex block
    r"\\end\s*\{",
    r"\\includegraphics",
    r"\[asy\]",             # diagram block marker before strip
]

# Main knobs for <= 3 hours. This is 40k answer-only, so 4 epochs is usually enough.
MAX_LENGTH = 256
GEN_MAX_NEW_TOKENS = 12
USE_SAMPLING_FALLBACK = False
NUM_SAMPLE_CANDIDATES = 3
GEN_TEMPERATURE = 0.7
GEN_TOP_P = 0.9

NUM_TRAIN_EPOCHS = 4
PER_DEVICE_TRAIN_BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 4e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.03

LOGGING_STEPS = 50
EVAL_STEPS = 500
SAVE_STEPS = 500
EVAL_LOSS_MAX_SAMPLES = 512
VALID_EVAL_MAX_SAMPLES = None  # None = full valid; set 300/500 for quick smoke test.

DO_TRAIN = True
DO_VALIDATE = True
DO_TEST_PREDICT = True

print("Variant:", VARIANT_NAME)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Train exclude types:", sorted(EXCLUDE_TRAIN_TYPES))
print("Target train samples:", TRAIN_MAX_SAMPLES)


In [ ]:


# %% [markdown]
# ## Cell 2 — I/O helpers and path discovery

# %%

def maybe_extract_local_data_zip() -> None:
    """Local debugging helper only. Kaggle normally reads directly from /kaggle/input."""
    zip_path = Path("/mnt/data/data.zip")
    extract_dir = Path("/mnt/data/extracted")
    if zip_path.exists() and not (extract_dir / "train.json").exists():
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)
        print(f"Extracted local data.zip to {extract_dir}")


maybe_extract_local_data_zip()


def first_existing(paths: List[Path], required: bool = True, name: str = "path") -> Optional[Path]:
    for p in paths:
        if p.exists():
            print(f"{name}: {p}")
            return p
    if required:
        raise FileNotFoundError(f"Cannot find {name}. Tried: {[str(p) for p in paths]}")
    print(f"{name}: not found")
    return None

TRAIN_PATH = first_existing(TRAIN_PATH_CANDIDATES, True, "TRAIN_PATH")
VALID_PATH = first_existing(VALID_PATH_CANDIDATES, True, "VALID_PATH")
TEST_PATH = first_existing(TEST_PATH_CANDIDATES, False, "TEST_PATH")


def find_model_path() -> Path:
    for p in MODEL_PATH_CANDIDATES:
        if p.exists():
            print("MODEL_PATH:", p)
            return p
    # Fallback: search common Kaggle input subdirs for config.json.
    root = Path("/kaggle/input")
    if root.exists():
        for cfg in root.rglob("config.json"):
            parent = cfg.parent
            files = {x.name for x in parent.iterdir() if x.is_file()}
            if any(name.startswith("pytorch_model") or name.endswith(".safetensors") for name in files):
                print("MODEL_PATH auto-found:", parent)
                return parent
    raise FileNotFoundError("Cannot find local NlpHUST/gpt2-vietnamese model directory.")


def read_json_or_jsonl(path: Path) -> Any:
    text = path.read_text(encoding="utf-8-sig").strip()
    if not text:
        return []
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # JSONL fallback.
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return rows


def ensure_list_records(obj: Any) -> List[Dict[str, Any]]:
    if isinstance(obj, list):
        return [x for x in obj if isinstance(x, dict)]
    if isinstance(obj, dict):
        for key in ["data", "records", "train", "valid", "test"]:
            if key in obj and isinstance(obj[key], list):
                return [x for x in obj[key] if isinstance(x, dict)]
    raise ValueError("Input JSON must be a list of objects or a dict containing a list field.")


def write_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def write_jsonl(rows: List[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

raw_train = ensure_list_records(read_json_or_jsonl(TRAIN_PATH))
raw_valid = ensure_list_records(read_json_or_jsonl(VALID_PATH))
print("raw_train:", len(raw_train), "raw_valid:", len(raw_valid))
print("raw_train type distribution:", dict(Counter(str(x.get("type", "UNKNOWN")) for x in raw_train)))
print("raw_valid type distribution:", dict(Counter(str(x.get("type", "UNKNOWN")) for x in raw_valid)))


# %% [markdown]
# ## Cell 3 — Answer extraction and scoring
# 
# Extractor này ưu tiên đáp án sau anchor (`Đáp án là`, `Câu trả lời là`, `The answer is`, `####`) và xử lý cơ bản `\frac{a}{b}`, `a/b`, `a\pi`, decimal comma, thousand separator.

# %%

ANSWER_ANCHOR_PATTERNS = [
    r"Đáp án là\s*[:：]?\s*([^\n]+)",
    r"Câu trả lời là\s*[:：]?\s*([^\n]+)",
    r"The answer is\s*[:：]?\s*([^\n]+)",
    r"####\s*([^\n]+)",
]

BAD_ANSWER_TOKENS = ["nan", "inf", "undefined", "không xác định", "không đủ", "unknown", "none", "n/a"]


def normalize_unicode_text(text: str) -> str:
    text = str(text or "")
    text = text.replace("\ufeff", "").replace("\u200b", "").replace("\u00a0", " ")
    text = text.replace("−", "-").replace("–", "-").replace("—", "-")
    text = text.replace("×", "*").replace("÷", "/")
    text = text.replace("\\đóng hộp", "\\boxed").replace("\\ đóng hộp", "\\boxed")
    text = text.replace("\\dfrac", "\\frac").replace("\\tfrac", "\\frac")
    return text


def strip_asy_blocks(text: str) -> str:
    return re.sub(r"\[asy\].*?\[/asy\]", " ", str(text or ""), flags=re.I | re.S)


def unwrap_boxed_once(s: str) -> str:
    m = re.search(r"\\boxed\s*\{", s)
    if not m:
        return s
    start = m.end()
    depth = 1
    i = start
    while i < len(s):
        if s[i] == "{":
            depth += 1
        elif s[i] == "}":
            depth -= 1
            if depth == 0:
                return s[:m.start()] + s[start:i] + s[i+1:]
        i += 1
    return s


def strip_boxed(s: str) -> str:
    s = str(s or "")
    for _ in range(5):
        s2 = unwrap_boxed_once(s)
        if s2 == s:
            break
        s = s2
    return s


def clean_answer_candidate(ans: str) -> str:
    ans = normalize_unicode_text(ans)
    ans = strip_boxed(ans).strip()
    ans = ans.replace("$", "").replace("`", "").strip()
    ans = re.split(r"\s+(?:Đáp án là|Câu trả lời là|The answer is)\s*[:：]?", ans, flags=re.I)[0]
    ans = ans.split("\n")[0].strip()
    ans = re.sub(r"^[=:：\s]+", "", ans).strip()
    ans = re.sub(r"(?:\.|,|;|:|。)+$", "", ans).strip()
    ans = re.sub(r"\s+", " ", ans)

    # Prefer latex fraction if present.
    m = re.search(r"-?\\frac\s*\{\s*-?\d+(?:[\.,]\d+)?\s*\}\s*\{\s*-?\d+(?:[\.,]\d+)?\s*\}", ans)
    if m:
        return m.group(0).replace(" ", "")

    # Prefer tuple if present.
    m = re.search(r"\((-?\d+(?:[\.,]\d+)?\s*,\s*)+-?\d+(?:[\.,]\d+)?\)", ans)
    if m:
        return re.sub(r"\s+", "", m.group(0))

    # Prefer coefficient*pi form.
    m = re.search(r"-?\d+(?:[\.,]\d+)?\s*\\?pi", ans, flags=re.I)
    if m:
        return m.group(0).replace(" ", "")

    # Prefer common fraction.
    m = re.search(r"-?\d+(?:[\.,]\d+)?\s*/\s*-?\d+(?:[\.,]\d+)?", ans)
    if m:
        return m.group(0).replace(" ", "")

    # Otherwise last numeric token, with optional thousand separators.
    nums = re.findall(r"-?\d+(?:[\.,]\d+)*", ans)
    if nums:
        return nums[-1]
    return ans.strip()


def extract_final_answer(text: str, fallback_last_number: bool = True) -> Optional[str]:
    text = normalize_unicode_text(text)
    if not text.strip():
        return None
    matches = []
    for pat in ANSWER_ANCHOR_PATTERNS:
        for m in re.finditer(pat, text, flags=re.I):
            matches.append((m.start(), m.group(1)))
    if matches:
        matches.sort(key=lambda x: x[0])
        cand = clean_answer_candidate(matches[-1][1])
        if cand and not any(tok in cand.lower() for tok in BAD_ANSWER_TOKENS):
            return cand
    if fallback_last_number:
        cand = clean_answer_candidate(text)
        if cand and not any(tok in cand.lower() for tok in BAD_ANSWER_TOKENS):
            return cand
    return None


def _normalize_num_token(x: str) -> str:
    x = str(x).strip().replace(" ", "")
    # 40.320 or 1,234,567 => thousand separators.
    if re.fullmatch(r"-?\d{1,3}([\.,]\d{3})+", x):
        return x.replace(".", "").replace(",", "")
    # 4,5 => 4.5 decimal comma.
    if re.fullmatch(r"-?\d+,\d{1,6}", x):
        return x.replace(",", ".")
    return x.replace(",", "")


def parse_numeric_answer(ans: Any) -> Optional[float]:
    if ans is None:
        return None
    s = clean_answer_candidate(str(ans))
    s = normalize_unicode_text(s)
    s = s.replace("$", "").strip()

    # Latex fraction.
    m = re.fullmatch(r"(-?)\\frac\{\s*(-?\d+(?:[\.,]\d+)?)\s*\}\{\s*(-?\d+(?:[\.,]\d+)?)\s*\}", s)
    if m:
        sign = -1.0 if m.group(1) == "-" else 1.0
        a = float(_normalize_num_token(m.group(2)))
        b = float(_normalize_num_token(m.group(3)))
        return sign * a / b if b != 0 else None

    # Common fraction.
    m = re.fullmatch(r"(-?\d+(?:[\.,]\d+)?)\s*/\s*(-?\d+(?:[\.,]\d+)?)", s)
    if m:
        a = float(_normalize_num_token(m.group(1)))
        b = float(_normalize_num_token(m.group(2)))
        return a / b if b != 0 else None

    # pi forms: pi, -pi/2, 36\pi.
    pi_s = s.replace("\\pi", "pi").replace("π", "pi")
    m = re.fullmatch(r"(-?)(?:(\d+(?:[\.,]\d+)?))?\s*pi(?:\s*/\s*(\d+(?:[\.,]\d+)?))?", pi_s, flags=re.I)
    if m:
        sign = -1.0 if m.group(1) == "-" else 1.0
        coef = float(_normalize_num_token(m.group(2))) if m.group(2) else 1.0
        den = float(_normalize_num_token(m.group(3))) if m.group(3) else 1.0
        return sign * coef * math.pi / den if den != 0 else None

    # Plain number.
    if re.fullmatch(r"-?\d+(?:[\.,]\d+)*", s):
        try:
            return float(_normalize_num_token(s))
        except Exception:
            return None
    return None


def relative_error(pred: Any, gold: Any) -> Optional[float]:
    p = parse_numeric_answer(pred)
    g = parse_numeric_answer(gold)
    if p is None or g is None:
        return None
    return abs(p - g) / max(1.0, abs(g))


def score_one(pred: Any, gold: Any) -> int:
    # Exact string fallback for non-scalar answers.
    if pred is not None and gold is not None:
        if clean_answer_candidate(str(pred)) == clean_answer_candidate(str(gold)):
            return 10
    err = relative_error(pred, gold)
    if err is None:
        return 0
    if err <= 0.01:
        return 10
    if err <= 0.10:
        return 5
    if err <= 0.50:
        return 1
    return 0

# Quick sanity checks.
for s in ["Đáp án là: 37", "Câu trả lời là: \\frac{9}{20}", "The answer is: 36\\pi", "#### 40.320"]:
    a = extract_final_answer(s)
    print(s, "=>", a, "=>", parse_numeric_answer(a))


# %% [markdown]
# ## Cell 4 — Variant 15 preprocessing: drop noisy train types, filter clean answers, select 40k
# 

# %%

def get_query(raw: Dict[str, Any]) -> str:
    return normalize_unicode_text(str(raw.get("query_vi", "") or "")).strip()


def get_response(raw: Dict[str, Any]) -> str:
    return normalize_unicode_text(str(raw.get("response_vi", "") or "")).strip()


def get_type(raw: Dict[str, Any]) -> str:
    return str(raw.get("type", "UNKNOWN") or "UNKNOWN")


def make_prompt(query: str) -> str:
    return f"Câu hỏi: {query}\nĐáp án:"


def make_target(answer: str) -> str:
    # Answer-only target: after the prompt, the model only learns the final answer token(s).
    return f" {answer}"


def normalize_query(q: str) -> str:
    q = normalize_unicode_text(q)
    q = re.sub(r"\s+", " ", q).strip()
    q = strip_asy_blocks(q)
    q = re.sub(r"\s+", " ", q).strip()
    return q


def is_heavy_latex_or_diagram_query(q: str) -> bool:
    original = normalize_unicode_text(q)
    return any(re.search(pat, original, flags=re.I | re.S) for pat in DROP_QUERY_REGEX)


def train_drop_reason(q: str, resp: str, typ: str, ans: Optional[str]) -> Optional[str]:
    if typ in EXCLUDE_TRAIN_TYPES:
        return f"excluded_type:{typ}"
    if not q:
        return "missing_query"
    if not resp:
        return "missing_response"
    if ans is None or not str(ans).strip():
        return "missing_answer"
    if len(q) > MAX_QUERY_CHARS:
        return "query_too_long"
    if len(str(ans)) > MAX_ANSWER_CHARS:
        return "answer_too_long"
    if any(tok in str(ans).lower() for tok in BAD_ANSWER_TOKENS):
        return "bad_answer_token"
    if is_heavy_latex_or_diagram_query(q):
        return "heavy_latex_or_diagram_query"
    if parse_numeric_answer(ans) is None:
        return "answer_not_numeric_parseable"
    return None


def valid_drop_reason(q: str, resp: str, typ: str, ans: Optional[str]) -> Optional[str]:
    # Keep validation as honest as possible. Do not drop hard types from valid.
    if not q:
        return "missing_query"
    if ans is None or not str(ans).strip():
        return "missing_answer"
    if any(tok in str(ans).lower() for tok in BAD_ANSWER_TOKENS):
        return "bad_answer_token"
    return None


def preprocess_records(raw_records: List[Dict[str, Any]], split: str, train_mode: bool = True) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    records = []
    drops = []
    seen_query_answer = set()
    query_to_answer = {}

    for i, raw in enumerate(raw_records):
        q_raw = get_query(raw)
        q = normalize_query(q_raw)
        resp = get_response(raw)
        typ = get_type(raw)
        ans = extract_final_answer(resp, fallback_last_number=False)
        ans = clean_answer_candidate(ans or "") if ans is not None else None

        reason = train_drop_reason(q_raw, resp, typ, ans) if train_mode else valid_drop_reason(q, resp, typ, ans)
        if reason is None:
            q_key = re.sub(r"\s+", " ", q.lower()).strip()
            a_key = clean_answer_candidate(ans or "")
            qa_key = (q_key, a_key)
            if train_mode and qa_key in seen_query_answer:
                reason = "duplicate_query_answer"
            elif train_mode and q_key in query_to_answer and query_to_answer[q_key] != a_key:
                reason = "duplicate_query_conflicting_answer"
            else:
                seen_query_answer.add(qa_key)
                query_to_answer[q_key] = a_key

        if reason:
            drops.append({
                "split": split,
                "raw_index": i,
                "reason": reason,
                "type": typ,
                "query_vi": q[:300],
                "answer": clean_answer_candidate(ans or ""),
            })
            continue

        rec = {
            "id": raw.get("id", i),
            "raw_index": i,
            "type": typ,
            "query_vi": q,
            "final_answer": clean_answer_candidate(ans),
            "prompt": make_prompt(q),
            "target": make_target(clean_answer_candidate(ans)),
        }
        records.append(rec)

    return records, drops


def select_balanced_clean_train(records: List[Dict[str, Any]], max_samples: int) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    rng = random.Random(SEED)
    by_type = defaultdict(list)
    for r in records:
        by_type[r["type"]].append(r)

    selected = []
    selected_raw = set()
    quota_report = []

    for typ, quota in PREFERRED_TYPE_QUOTAS.items():
        pool = list(by_type.get(typ, []))
        rng.shuffle(pool)
        take = pool[:min(quota, len(pool))]
        selected.extend(take)
        selected_raw.update(r["raw_index"] for r in take)
        quota_report.append({"type": typ, "quota": quota, "available": len(pool), "taken": len(take)})

    # Backfill if filtering makes any quota short.
    if len(selected) < max_samples:
        remaining = [r for r in records if r["raw_index"] not in selected_raw]
        rng.shuffle(remaining)
        need = max_samples - len(selected)
        selected.extend(remaining[:need])

    if len(selected) > max_samples:
        rng.shuffle(selected)
        selected = selected[:max_samples]

    rng.shuffle(selected)
    report = {
        "requested_max_samples": max_samples,
        "selected_samples": len(selected),
        "quota_report": quota_report,
        "selected_type_distribution": dict(Counter(r["type"] for r in selected)),
        "candidate_type_distribution": dict(Counter(r["type"] for r in records)),
    }
    return selected, report


train_candidate_records, train_drops = preprocess_records(raw_train, "train", train_mode=True)
valid_records, valid_drops = preprocess_records(raw_valid, "valid", train_mode=False)
train_records, selection_report = select_balanced_clean_train(train_candidate_records, TRAIN_MAX_SAMPLES)

write_jsonl(train_records, PROC_DIR / "train_answer_only_clean40k.jsonl")
write_jsonl(valid_records, PROC_DIR / "valid_answer_only.jsonl")
write_jsonl(train_drops + valid_drops, PROC_DIR / "drop_log.jsonl")

prep_report = {
    "variant": VARIANT_NAME,
    "target_format": "answer_only",
    "train_policy": {
        "excluded_train_types": sorted(EXCLUDE_TRAIN_TYPES),
        "preferred_type_quotas": PREFERRED_TYPE_QUOTAS,
        "max_query_chars": MAX_QUERY_CHARS,
        "max_answer_chars": MAX_ANSWER_CHARS,
        "drop_query_regex": DROP_QUERY_REGEX,
    },
    "counts": {
        "raw_train": len(raw_train),
        "raw_valid": len(raw_valid),
        "train_candidates_after_filter": len(train_candidate_records),
        "train_records_selected": len(train_records),
        "valid_records": len(valid_records),
        "train_dropped": len(train_drops),
        "valid_dropped": len(valid_drops),
    },
    "raw_train_type_distribution": dict(Counter(get_type(r) for r in raw_train)),
    "candidate_train_type_distribution": dict(Counter(r["type"] for r in train_candidate_records)),
    "selected_train_type_distribution": dict(Counter(r["type"] for r in train_records)),
    "valid_type_distribution": dict(Counter(r["type"] for r in valid_records)),
    "train_drop_summary": dict(Counter(d["reason"] for d in train_drops)),
    "valid_drop_summary": dict(Counter(d["reason"] for d in valid_drops)),
    "selection_report": selection_report,
}
write_json(prep_report, PROC_DIR / "preprocess_report.json")
print(json.dumps(prep_report, ensure_ascii=False, indent=2)[:5000])

print("Selected train size:", len(train_records))
print("Selected train type distribution:", dict(Counter(r["type"] for r in train_records)))
display(pd.DataFrame(train_records[:8])[["type", "query_vi", "final_answer", "prompt", "target"]])

# Small audit table for what was removed.
if train_drops:
    drop_df = pd.DataFrame(train_drops)
    display(drop_df.groupby(["type", "reason"]).size().reset_index(name="n").sort_values("n", ascending=False).head(30))


# %% [markdown]
# ## Cell 5 — Load tokenizer/model and build torch datasets

# %%

from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = find_model_path()

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(str(MODEL_PATH), local_files_only=True)

# Force safe eos/pad ids required by the competition note.
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token is not None else "<|endoftext|>"
model.config.pad_token_id = SAFE_EOS_ID
model.config.eos_token_id = SAFE_EOS_ID

print("vocab_size:", len(tokenizer), "pad:", tokenizer.pad_token_id, "eos:", tokenizer.eos_token_id)

class AnswerOnlyDataset(Dataset):
    def __init__(self, records: List[Dict[str, Any]], tokenizer, max_length: int):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        r = self.records[idx]
        prompt = r["prompt"]
        target = r["target"] + (self.tokenizer.eos_token or "")

        prompt_ids = self.tokenizer(prompt, add_special_tokens=False)["input_ids"]
        full = self.tokenizer(
            prompt + target,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_length,
        )
        input_ids = full["input_ids"]
        attention_mask = full["attention_mask"]
        labels = input_ids.copy()
        prompt_len = min(len(prompt_ids), len(labels))
        labels[:prompt_len] = [-100] * prompt_len

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

class CausalLMCollator:
    def __init__(self, tokenizer, label_pad_token_id: int = -100):
        self.tokenizer = tokenizer
        self.label_pad_token_id = label_pad_token_id

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            batch["input_ids"].append(torch.cat([f["input_ids"], torch.full((pad_len,), self.tokenizer.pad_token_id, dtype=torch.long)]))
            batch["attention_mask"].append(torch.cat([f["attention_mask"], torch.zeros(pad_len, dtype=torch.long)]))
            batch["labels"].append(torch.cat([f["labels"], torch.full((pad_len,), self.label_pad_token_id, dtype=torch.long)]))
        return {k: torch.stack(v) for k, v in batch.items()}

train_dataset = AnswerOnlyDataset(train_records, tokenizer, MAX_LENGTH)
eval_loss_records = valid_records[:min(EVAL_LOSS_MAX_SAMPLES, len(valid_records))]
eval_dataset = AnswerOnlyDataset(eval_loss_records, tokenizer, MAX_LENGTH)
collator = CausalLMCollator(tokenizer)

print("train_dataset:", len(train_dataset), "eval_dataset:", len(eval_dataset))


# %% [markdown]
# ## Cell 6 — Apply LoRA

# %%

try:
    from peft import LoraConfig, get_peft_model, TaskType
except Exception as e:
    raise ImportError(
        "PEFT is required for this LoRA variant. On Kaggle, attach a dataset/package with peft if needed, still with Internet OFF."
    ) from e

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["c_attn", "c_proj", "c_fc"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if torch.cuda.is_available():
    model = model.cuda()


# %% [markdown]
# ## Cell 7 — Fine-tune answer-only LoRA

# %%

from transformers import Trainer, TrainingArguments


def build_training_args() -> TrainingArguments:
    raw_kwargs = dict(
        output_dir=str(MODEL_OUT_DIR),
        overwrite_output_dir=True,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        eval_steps=EVAL_STEPS,
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        report_to="none",
        dataloader_num_workers=2,
        remove_unused_columns=False,
        load_best_model_at_end=False,
    )
    sig = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in sig:
        raw_kwargs["eval_strategy"] = "steps"
    elif "evaluation_strategy" in sig:
        raw_kwargs["evaluation_strategy"] = "steps"

    kwargs = {k: v for k, v in raw_kwargs.items() if k in sig}
    dropped = sorted(set(raw_kwargs) - set(kwargs))
    if dropped:
        print("Dropped unsupported TrainingArguments:", dropped)
    return TrainingArguments(**kwargs)


def build_trainer_kwargs(args: TrainingArguments) -> Dict[str, Any]:
    base = dict(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=collator,
    )
    sig = inspect.signature(Trainer.__init__).parameters
    return {k: v for k, v in base.items() if k in sig}

if DO_TRAIN:
    training_args = build_training_args()
    trainer = Trainer(**build_trainer_kwargs(training_args))
    train_result = trainer.train()
    print(train_result)
    trainer.save_model(str(MODEL_OUT_DIR))
    tokenizer.save_pretrained(str(MODEL_OUT_DIR))
    write_json({"train_result": str(train_result), "variant": VARIANT_NAME}, MODEL_OUT_DIR / "train_result.json")
else:
    print("DO_TRAIN=False, skip training")

model.eval()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# %% [markdown]
# ## Cell 8 — Inference policy: model-only, answer-only
# 
# Notebook không định nghĩa rule solver/Python solver từ đề. Inference chỉ generate bằng model, extract đáp án từ output, rồi format thành `Đáp án là: <answer>`.
# 

# %%

INFERENCE_POLICY = {
    "variant": VARIANT_NAME,
    "model_only": True,
    "answer_only": True,
    "no_rule_solver": True,
    "no_python_math_solver_from_query": True,
    "allowed_python_usage": [
        "data_loading",
        "data_filtering_and_selection",
        "answer_extraction_from_gold_or_model_output",
        "local_validation_scoring",
        "json_export",
    ],
    "candidate_sources": ["model_greedy", "model_sample"] if USE_SAMPLING_FALLBACK else ["model_greedy"],
}
print(json.dumps(INFERENCE_POLICY, ensure_ascii=False, indent=2))


# %% [markdown]
# ## Cell 9 — Model generation, model-only voting and final prediction functions
# 

# %%

@torch.no_grad()
def generate_answer_only(prompt: str, do_sample: bool = False, num_return_sequences: int = 1) -> List[str]:
    model.eval()
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(model.device)

    gen_kwargs = dict(
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        pad_token_id=SAFE_EOS_ID,
        eos_token_id=SAFE_EOS_ID,
        num_return_sequences=num_return_sequences,
    )
    if do_sample:
        gen_kwargs.update(dict(do_sample=True, temperature=GEN_TEMPERATURE, top_p=GEN_TOP_P))
    else:
        gen_kwargs.update(dict(do_sample=False, num_beams=1))

    out = model.generate(**inputs, **gen_kwargs)
    decoded = tokenizer.batch_decode(out, skip_special_tokens=True)

    answers = []
    for text in decoded:
        # Remove prompt prefix if model repeats it. This is not solving the problem;
        # it only extracts the answer generated by the model.
        tail = text[len(prompt):] if text.startswith(prompt) else text.split("Đáp án:")[-1]
        ans = extract_final_answer(tail, fallback_last_number=True)
        if ans is None:
            ans = clean_answer_candidate(tail)
        ans = clean_answer_candidate(ans)
        if ans:
            answers.append(ans)
    return answers


def canonical_answer_key(ans: str) -> str:
    ans = clean_answer_candidate(ans)
    val = parse_numeric_answer(ans)
    if val is not None and math.isfinite(float(val)):
        return f"num:{float(val):.10g}"
    return f"str:{ans}"


def choose_answer_from_candidates(cands: List[str]) -> Tuple[Optional[str], Dict[str, Any]]:
    cleaned = [clean_answer_candidate(c) for c in cands if c and clean_answer_candidate(c)]
    if not cleaned:
        return None, {"source": "none", "candidates": []}

    grouped = defaultdict(list)
    for c in cleaned:
        grouped[canonical_answer_key(c)].append(c)

    # Prefer the most common canonical answer; tie-break by shorter representative.
    best_key, best_group = sorted(
        grouped.items(),
        key=lambda kv: (-len(kv[1]), min(len(x) for x in kv[1]), kv[0]),
    )[0]
    representative = sorted(best_group, key=lambda x: (len(x), x))[0]
    return representative, {
        "source": "model_vote",
        "candidates": cleaned,
        "canonical_counts": {k: len(v) for k, v in grouped.items()},
        "best_key": best_key,
    }


def make_model_output(answer: str, source: str = "model") -> str:
    answer = clean_answer_candidate(answer or "")
    if not answer:
        # Last-resort formatting fallback only. It does not inspect the query or solve with rules.
        answer = "0"
    return f"Đáp án là: {answer}"


def predict_one(query: str) -> Tuple[str, Dict[str, Any]]:
    prompt = make_prompt(normalize_query(query))

    all_candidates = []
    source_trace = []

    # 1) Deterministic model generation.
    greedy_cands = generate_answer_only(prompt, do_sample=False, num_return_sequences=1)
    all_candidates.extend(greedy_cands)
    source_trace.append({"source": "model_greedy", "candidates": greedy_cands})

    # 2) Optional model-only sampling fallback. Every candidate is still generated by the model.
    if USE_SAMPLING_FALLBACK:
        sample_cands = generate_answer_only(prompt, do_sample=True, num_return_sequences=NUM_SAMPLE_CANDIDATES)
        all_candidates.extend(sample_cands)
        source_trace.append({"source": "model_sample", "candidates": sample_cands})

    ans, info = choose_answer_from_candidates(all_candidates)
    if ans is not None:
        return make_model_output(ans, source="model_vote"), {
            "source": "model_vote",
            "answer": ans,
            "trace": source_trace,
            **info,
        }

    return make_model_output("0", source="fallback_format_only"), {
        "source": "fallback_format_only",
        "answer": "0",
        "trace": source_trace,
    }


# %% [markdown]
# ## Cell 10 — Validation inference + report

# %%

def evaluate_records(records: List[Dict[str, Any]], max_samples: Optional[int] = None) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    eval_records = list(records)
    if max_samples is not None and len(eval_records) > max_samples:
        rng = random.Random(SEED)
        eval_records = rng.sample(eval_records, max_samples)

    outputs = []
    for i, r in enumerate(eval_records):
        if i % 50 == 0:
            print(f"Evaluating {i}/{len(eval_records)}")
        model_output, info = predict_one(r["query_vi"])
        pred_answer = extract_final_answer(model_output, fallback_last_number=True)
        gold_answer = r["final_answer"]
        sc = score_one(pred_answer, gold_answer)
        err = relative_error(pred_answer, gold_answer)
        outputs.append({
            "id": r.get("id", i),
            "raw_index": r.get("raw_index", i),
            "query_vi": r["query_vi"],
            "type": r["type"],
            "gold_answer": gold_answer,
            "pred_answer": clean_answer_candidate(pred_answer or ""),
            "relative_error": err,
            "score": sc,
            "model_output": model_output,
            "predict_info": info,
        })

    n = len(outputs)
    report = {
        "variant": VARIANT_NAME,
        "n": n,
        "raw_score": int(sum(x["score"] for x in outputs)),
        "score_10": float(sum(x["score"] for x in outputs) / n) if n else 0.0,
        "exact_10_count": int(sum(x["score"] == 10 for x in outputs)),
        "score_5_count": int(sum(x["score"] == 5 for x in outputs)),
        "score_1_count": int(sum(x["score"] == 1 for x in outputs)),
        "score_0_count": int(sum(x["score"] == 0 for x in outputs)),
        "extract_rate": float(np.mean([bool(x["pred_answer"]) for x in outputs])) if n else 0.0,
        "source_counts": dict(Counter(x["predict_info"].get("source", "unknown") for x in outputs)),
        "inference_policy": INFERENCE_POLICY,
    }
    return outputs, report

if DO_VALIDATE:
    valid_outputs, valid_report = evaluate_records(valid_records, VALID_EVAL_MAX_SAMPLES)
    write_json(valid_outputs, WORK_DIR / "valid_output.json")
    write_json(valid_report, WORK_DIR / "valid_report.json")
    print(json.dumps(valid_report, ensure_ascii=False, indent=2))
else:
    valid_outputs, valid_report = [], {}
    print("DO_VALIDATE=False, skip validation")


# %% [markdown]
# ## Cell 11 — Report by type and error table

# %%

if valid_outputs:
    df_eval = pd.DataFrame(valid_outputs)

    def summarize_group(g: pd.DataFrame) -> pd.Series:
        return pd.Series({
            "n": len(g),
            "score_10": g["score"].sum() / len(g) if len(g) else 0,
            "extract_rate": (g["pred_answer"].astype(str).str.len() > 0).mean() if len(g) else 0,
            "score_10_count": int((g["score"] == 10).sum()),
            "score_5_count": int((g["score"] == 5).sum()),
            "score_1_count": int((g["score"] == 1).sum()),
            "score_0_count": int((g["score"] == 0).sum()),
        })

    type_report = df_eval.groupby("type").apply(summarize_group).reset_index()
    type_report = type_report.sort_values(["score_10", "n"], ascending=[True, False])
    display(type_report)
    type_report.to_csv(WORK_DIR / "valid_report_by_type.csv", index=False, encoding="utf-8-sig")

    debug_cols = ["raw_index", "type", "score", "relative_error", "gold_answer", "pred_answer", "query_vi", "model_output", "predict_info"]
    errors_df = df_eval[df_eval["score"] < 10].sort_values(["score", "relative_error"], ascending=[True, False], na_position="last")
    display(errors_df[debug_cols].head(30))
    errors_df[debug_cols].to_csv(WORK_DIR / "valid_errors.csv", index=False, encoding="utf-8-sig")

    full_summary = {"overall": valid_report, "by_type": type_report.to_dict(orient="records"), "preprocess": prep_report}
    write_json(full_summary, WORK_DIR / "valid_full_summary.json")
else:
    print("No valid outputs to report.")


# %% [markdown]
# ## Cell 12 — Generate `test_predictions.json` when test exists

# %%

def build_test_records(raw_test: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    records = []
    for i, raw in enumerate(raw_test):
        q = normalize_query(str(raw.get("query_vi", "") or ""))
        records.append({
            "id": raw.get("id", i),
            "query_vi": q,
            "type": str(raw.get("type", "UNKNOWN") or "UNKNOWN"),
        })
    return records


def predict_test(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    preds = []
    for i, r in enumerate(records):
        if i % 50 == 0:
            print(f"Predicting {i}/{len(records)}")
        model_output, info = predict_one(r["query_vi"])
        preds.append({
            "id": r["id"],
            "query_vi": r["query_vi"],
            "type": r["type"],
            "model_output": model_output,
        })
    return preds

if DO_TEST_PREDICT and TEST_PATH is not None and TEST_PATH.exists():
    print("Found test file:", TEST_PATH)
    raw_test = ensure_list_records(read_json_or_jsonl(TEST_PATH))
    test_records = build_test_records(raw_test)
    test_predictions = predict_test(test_records)
    write_json(test_predictions, WORK_DIR / "test_predictions.json")
    print("Saved:", WORK_DIR / "test_predictions.json", "n=", len(test_predictions))
elif DO_TEST_PREDICT:
    # Smoke-test output format using first 2 valid examples so the notebook always leaves an example file.
    print("No official test.json found. Creating sample_test_predictions_from_valid.json for format check only.")
    sample_records = [
        {"id": r["id"], "query_vi": r["query_vi"], "type": r["type"]}
        for r in valid_records[:2]
    ]
    sample_predictions = predict_test(sample_records)
    write_json(sample_predictions, WORK_DIR / "sample_test_predictions_from_valid.json")
    display(pd.DataFrame(sample_predictions))
else:
    print("DO_TEST_PREDICT=False, skip test prediction")


# %% [markdown]
# ## Cell 13 — Final artifact list

# %%

print("Important outputs:")
for p in [
    PROC_DIR / "preprocess_report.json",
    PROC_DIR / "train_answer_only_clean40k.jsonl",
    PROC_DIR / "drop_log.jsonl",
    WORK_DIR / "valid_output.json",
    WORK_DIR / "valid_report.json",
    WORK_DIR / "valid_report_by_type.csv",
    WORK_DIR / "valid_errors.csv",
    WORK_DIR / "test_predictions.json",
    WORK_DIR / "sample_test_predictions_from_valid.json",
    MODEL_OUT_DIR,
]:
    print("-", p, "exists=", p.exists())

Variant: v15_answer_only_drop_math_ansaug_clean40k
CUDA: True
GPU: Tesla T4
Train exclude types: ['GSM_FOBAR', 'MATH_AnsAug', 'MATH_FOBAR', 'MATH_SV']
Target train samples: 40000
TRAIN_PATH: /kaggle/input/datasets/kimanh2002/dataset-math/train.json
VALID_PATH: /kaggle/input/datasets/kimanh2002/dataset-math/valid.json
TEST_PATH: not found
raw_train: 95400 raw_valid: 1000
raw_train type distribution: {'GSM_AnsAug': 18745, 'MATH_AnsAug': 16999, 'GSM_SV': 9869, 'GSM_FOBAR': 10023, 'MATH_Rephrased': 12477, 'MATH_SV': 3591, 'GSM_Rephrased': 20028, 'MATH_FOBAR': 3668}
raw_valid type distribution: {'GSM_Rephrased': 197, 'MATH_Rephrased': 116, 'MATH_SV': 41, 'GSM_AnsAug': 209, 'GSM_SV': 97, 'GSM_FOBAR': 122, 'MATH_AnsAug': 173, 'MATH_FOBAR': 45}
Đáp án là: 37 => 37 => 37.0
Câu trả lời là: \frac{9}{20} => \frac{9}{20} => 0.45
The answer is: 36\pi => 36\pi => 113.09733552923255
#### 40.320 => 40.320 => 40320.0
{
  "variant": "v15_answer_only_drop_math_ansaug_clean40k",
  "target_format": "answer_

,type,query_vi,final_answer,prompt,target
0,GSM_SV,"Vào thứ Tư, 37 học sinh đã chơi đá bóng. Vào t...",9,"Câu hỏi: Vào thứ Tư, 37 học sinh đã chơi đá bó...",9
1,GSM_SV,Tom chỉ ngủ được x giờ mỗi ngày. Anh ta tăng s...,6,Câu hỏi: Tom chỉ ngủ được x giờ mỗi ngày. Anh ...,6
2,GSM_SV,Marnie làm vòng tay bằng hạt. Cô mua 5 túi 50 ...,2,Câu hỏi: Marnie làm vòng tay bằng hạt. Cô mua ...,2
3,GSM_AnsAug,Joey có 30 trang để đọc cho lớp lịch sử tối na...,9,Câu hỏi: Joey có 30 trang để đọc cho lớp lịch ...,9
4,GSM_Rephrased,Một người bán hàng mua một hộp đựng 48 chiếc b...,442,Câu hỏi: Một người bán hàng mua một hộp đựng 4...,442
5,GSM_Rephrased,"Trong một cuộc thi có 40 người tham gia, một s...",96,Câu hỏi: Trong một cuộc thi có 40 người tham g...,96
6,GSM_SV,"Trong một chuyến đi học ở bờ biển, Alan và các...",4,"Câu hỏi: Trong một chuyến đi học ở bờ biển, Al...",4
7,MATH_Rephrased,Nếu số nguyên dương $N$ có ba chữ số có một ch...,\frac{1}{3},Câu hỏi: Nếu số nguyên dương $N$ có ba chữ số ...,\frac{1}{3}


,type,reason,n
7,MATH_AnsAug,excluded_type:MATH_AnsAug,16999
0,GSM_AnsAug,duplicate_query_answer,12437
2,GSM_FOBAR,excluded_type:GSM_FOBAR,10023
10,MATH_Rephrased,duplicate_query_answer,3813
8,MATH_FOBAR,excluded_type:MATH_FOBAR,3668
14,MATH_SV,excluded_type:MATH_SV,3591
3,GSM_Rephrased,duplicate_query_answer,2972
4,GSM_SV,duplicate_query_answer,671
12,MATH_Rephrased,heavy_latex_or_diagram_query,283
9,MATH_Rephrased,answer_not_numeric_parseable,242


MODEL_PATH: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

,type,n,score_10,extract_rate,score_10_count,score_5_count,score_1_count,score_0_count
5,MATH_FOBAR,45.0,0.977778,1.0,3.0,1.0,9.0,32.0
0,GSM_AnsAug,209.0,1.052632,1.0,11.0,9.0,65.0,124.0
4,MATH_AnsAug,171.0,1.064327,1.0,12.0,4.0,42.0,113.0
6,MATH_Rephrased,115.0,1.495652,1.0,13.0,4.0,22.0,76.0
3,GSM_SV,97.0,1.762887,1.0,13.0,1.0,36.0,47.0
1,GSM_FOBAR,122.0,1.926230,1.0,20.0,0.0,35.0,67.0
2,GSM_Rephrased,197.0,2.045685,1.0,28.0,11.0,68.0,90.0
7,MATH_SV,41.0,2.634146,1.0,9.0,2.0,8.0,22.0


,raw_index,type,score,relative_error,gold_answer,pred_answer,query_vi,model_output,predict_info
101,101,MATH_Rephrased,0,374.100000,20,7502,"Từ một nhóm 5 người, có bao nhiêu cách chọn Ch...",Đáp án là: 7502,"{'source': 'model_vote', 'answer': '7502', 'tr..."
147,147,GSM_AnsAug,0,149.000000,10,1500,Nancy muốn tìm hiểu xem liệu cô ấy có đủ khả n...,Đáp án là: 1500,"{'source': 'model_vote', 'answer': '1500', 'tr..."
558,561,GSM_AnsAug,0,107.333333,3,325,Johnny đã viết một bài luận khoảng 150 từ. Mad...,Đáp án là: 325,"{'source': 'model_vote', 'answer': '325', 'tra..."
171,171,MATH_FOBAR,0,99.000000,1,100,"Giả sử $F_n$ là số Fibonacci thứ $n$, trong đó...",Đáp án là: 100,"{'source': 'model_vote', 'answer': '100', 'tra..."
266,267,MATH_SV,0,99.000000,10,1000,Hai dãy số học $A$ và $B$ đều bắt đầu bằng 30 ...,Đáp án là: 1000,"{'source': 'model_vote', 'answer': '1000', 'tr..."
685,688,GSM_FOBAR,0,74.000000,2,150,Một bà nội trợ đi chợ. Cô ấy đã tiêu x trong s...,Đáp án là: 150,"{'source': 'model_vote', 'answer': '150', 'tra..."
108,108,GSM_Rephrased,0,67.181818,22,1500,Nếu một nhà xuất bản trao tổng giải thưởng là ...,Đáp án là: 1500,"{'source': 'model_vote', 'answer': '1500', 'tr..."
176,176,MATH_SV,0,50.000000,0,50,Giải bất đẳng thức \[\frac{x^2 - 25}{x + 5} < ...,Đáp án là: 50,"{'source': 'model_vote', 'answer': '50', 'trac..."
971,974,MATH_AnsAug,0,38.500000,2,-75,Phân tích nhân tử của biểu thức sau: $45x+30$.,Đáp án là: -75,"{'source': 'model_vote', 'answer': '-75', 'tra..."
494,497,GSM_AnsAug,0,37.461538,24960,960000,Trisha vừa nhận được lời mời làm việc theo giờ...,Đáp án là: 960000,"{'source': 'model_vote', 'answer': '960000', '..."


No official test.json found. Creating sample_test_predictions_from_valid.json for format check only.
Predicting 0/2


,id,query_vi,type,model_output
0,0,Nếu Susan đang chơi một trò chơi cờ bàn có 48 ...,GSM_Rephrased,Đáp án là: 5
1,1,"Nếu $\angle PQR = \angle PRQ$, và độ dài của Q...",MATH_Rephrased,Đáp án là: 30


Important outputs:
- /kaggle/working/processed_v15_answer_only_drop_math_ansaug_clean40k/preprocess_report.json exists= True
- /kaggle/working/processed_v15_answer_only_drop_math_ansaug_clean40k/train_answer_only_clean40k.jsonl exists= True
- /kaggle/working/processed_v15_answer_only_drop_math_ansaug_clean40k/drop_log.jsonl exists= True
- /kaggle/working/valid_output.json exists= True
- /kaggle/working/valid_report.json exists= True
- /kaggle/working/valid_report_by_type.csv exists= True
- /kaggle/working/valid_errors.csv exists= True
- /kaggle/working/test_predictions.json exists= False
- /kaggle/working/sample_test_predictions_from_valid.json exists= True
- /kaggle/working/lora_v15_answer_only_drop_math_ansaug_clean40k exists= True
